# Bonus Challenge 5 — Deploying Agents



In [ ]:
# =============================================================================
# SETUP & AGENT DEFINITION
# =============================================================================
# %pip install -q "google-cloud-aiplatform[adk,agent_engines]" google-adk

import os

import vertexai
from google.adk.agents import Agent, SequentialAgent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool
from google.cloud import storage
from vertexai import agent_engines

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")  # auto-set in Colab Enterprise
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")
BUCKET_NAME = f"{PROJECT_ID}-challenge5-staging"
STAGING_BUCKET = f"gs://{BUCKET_NAME}"
DISPLAY_NAME = "challenge5-answer-team"
MODEL = "gemini-2.5-flash"

assert PROJECT_ID, "GOOGLE_CLOUD_PROJECT not found — set PROJECT_ID manually."

vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)

# vertexai.init does not create the bucket.
_sc = storage.Client(project=PROJECT_ID)
if _sc.lookup_bucket(BUCKET_NAME) is None:
    _sc.create_bucket(BUCKET_NAME, location=LOCATION)
    print(f"Created staging bucket {STAGING_BUCKET}")

# ---------------------------------------------------------------------------
# Search agent: google_search is a built-in and cannot share an agent with
# other tools, so it lives alone and is exposed via AgentTool.
# ---------------------------------------------------------------------------
search_agent = Agent(
    name="SearchAgent",
    model=MODEL,
    description="Finds current information on the web using Google Search.",
    instruction=(
        "Use google_search to find current, accurate information for the request. "
        "Prefer primary and official sources. Report what you actually found, and "
        "say plainly when you could not confirm something."
    ),
    tools=[google_search],
)

# ---------------------------------------------------------------------------
# Sequential answer team: draft -> critique -> refine
# ---------------------------------------------------------------------------
answerer = Agent(
    name="Answerer",
    model=MODEL,
    description="Drafts an answer, calling the search agent when facts are needed.",
    instruction=(
        "Answer the user's question. Call SearchAgent whenever the answer depends on "
        "current facts, figures, or anything that may have changed recently. "
        "State plainly when something could not be verified."
    ),
    tools=[AgentTool(agent=search_agent)],
    output_key="draft_answer",
)

critic = Agent(
    name="Critic",
    model=MODEL,
    description="Reviews the draft and suggests concrete improvements.",
    instruction=(
        "Review this draft answer:\n\n{draft_answer}\n\n"
        "List concrete, numbered improvements only. Check for: claims with no source; "
        "vague or hedged wording; missing context the reader needs; and anything that "
        "does not actually answer the question asked. "
        "If the draft is sound, reply exactly: NO ISSUES."
    ),
    output_key="critique",
)

refiner = Agent(
    name="Refiner",
    model=MODEL,
    description="Rewrites the draft applying the critique. Produces the final answer.",
    instruction=(
        "Rewrite this draft:\n\n{draft_answer}\n\nApplying this critique:\n\n{critique}\n\n"
        "If the critique is 'NO ISSUES', return the draft with formatting cleaned up only. "
        "Output the final answer directly to the user. Do not mention the draft, the "
        "critique, or this process."
    ),
    output_key="final_answer",
)

answer_team = SequentialAgent(
    name="AnswerTeam",
    description="Answers a question, then validates and refines the answer before returning it.",
    sub_agents=[answerer, critic, refiner],
)

# ---------------------------------------------------------------------------
# Greeter root agent
# ---------------------------------------------------------------------------
greeter = Agent(
    name="Greeter",
    model=MODEL,
    description="Entry point. Greets the user and routes questions to the answer team.",
    instruction=(
        "You are a helpful research assistant. If the user only greets you or asks what "
        "you can do, reply directly and briefly. For any actual question, transfer to "
        "AnswerTeam, which drafts, critiques, and refines the answer."
    ),
    sub_agents=[answer_team],
)

root_agent = greeter  # ADK convention
app = agent_engines.AdkApp(agent=root_agent, enable_tracing=True)

print("Agent built.")
print(f"  Project: {PROJECT_ID} ({LOCATION})")
print(f"  {root_agent.name} -> {answer_team.name} -> {[a.name for a in answer_team.sub_agents]}")


In [ ]:
# =============================================================================
# LOCAL TEST
# =============================================================================


import time


def _session_id(session):
    """create_session returns a dict in current SDK versions."""
    return session["id"] if isinstance(session, dict) else session.id


def run_query(runner, label, message, user_id, show_events=True, retries=3):
    print("=" * 78)
    print(f"{label}\nUSER: {message}")
    print("-" * 78)
    final = None
    for attempt in range(retries):
        session = runner.create_session(user_id=user_id)
        final = None
        try:
            for event in runner.stream_query(
                user_id=user_id, session_id=_session_id(session), message=message
            ):
                author = event.get("author", "?")
                for part in (event.get("content") or {}).get("parts", []) or []:
                    if show_events and part.get("function_call"):
                        print(f"   [{author}] CALL   {part['function_call'].get('name')}")
                    if part.get("text"):
                        print(f"   [{author}] {part['text'].strip()[:180]}")
                        final = part["text"]
        except Exception as exc:
            print(f"   [run] attempt {attempt + 1} raised: {type(exc).__name__}")
        if final:
            break
        # A quota 429 raises on ADK's background thread, so it reaches us as an
        # empty stream rather than an exception. Back off and retry.
        if attempt < retries - 1:
            wait = 20 * (attempt + 1)
            print(f"   [run] no output - likely quota (429); retrying in {wait}s")
            time.sleep(wait)
    print("-" * 78)
    print("FINAL:\n" + (final or "(no text returned after retries)"))
    print()
    return final


# Fresh session per query: after transferring into the SequentialAgent, the last
# agent in the sequence stays active for follow-up turns.
run_query(app, "LOCAL TEST 1 - greeting (root answers directly)",
          "Hi, what can you do?", user_id="local-test")
time.sleep(10)
run_query(app, "LOCAL TEST 2 - full workflow (search -> critique -> refine)",
          "What is Google's Agent Development Kit and what is it used for?",
          user_id="local-test")


In [ ]:
# =============================================================================
# DEPLOY TO AGENT ENGINE + REMOTE TEST
# =============================================================================
#

from importlib import metadata


def pinned_requirements():
    """Pin to this kernel's versions — skew between notebook and container is the
    most common cause of a deploy that builds but fails at runtime."""
    pins = []
    for pkg in ("google-cloud-aiplatform", "google-adk"):
        try:
            pins.append(f"{pkg}=={metadata.version(pkg)}")
        except metadata.PackageNotFoundError:
            pins.append(pkg)
    pins[0] = pins[0].replace(
        "google-cloud-aiplatform==", "google-cloud-aiplatform[agent_engines,adk]=="
    )
    return pins


existing = next(
    (e for e in agent_engines.list() if getattr(e, "display_name", None) == DISPLAY_NAME),
    None,
)

if existing is not None:
    print(f"Updating existing engine: {existing.resource_name}")
    remote_app = existing.update(
        agent_engine=app, requirements=pinned_requirements(), display_name=DISPLAY_NAME
    )
else:
    print("Creating new Agent Engine instance (5-10 min)...")
    remote_app = agent_engines.create(
        app,
        display_name=DISPLAY_NAME,
        description="Challenge 5: Greeter + Sequential answer team (search, critique, refine).",
        requirements=pinned_requirements(),
    )

print(f"\nDeployed: {remote_app.resource_name}")

# If this cell errors with a 500 while POLLING, the build is still running
# server-side. Do NOT re-run create — wait, then rebind with:
#   remote_app = agent_engines.get("<resource name printed above>")


# ---------------------------------------------------------------------------
# Test the DEPLOYED agent
# ---------------------------------------------------------------------------
run_query(remote_app, "REMOTE TEST 1 - greeting", "Hi, what can you do?", user_id="remote-test")
time.sleep(20)  # stay under the per-minute model quota
run_query(remote_app, "REMOTE TEST 2 - full workflow",
          "What is Google's Agent Development Kit and what is it used for?",
          user_id="remote-test")

print("=" * 78)
print("AGENT PLAYGROUND")
print("=" * 78)
print(f"https://console.cloud.google.com/vertex-ai/agents/agent-engines?project={PROJECT_ID}")
print(f"  Open the instance '{DISPLAY_NAME}' -> Playground tab to chat with it.")
print("  The Trace tab shows the sub-agent calls for each turn (enable_tracing=True).")
print(f"  Resource name: {remote_app.resource_name}")
print()
print("  CLEANUP after grading (engines bill while they exist):")
print("    remote_app.delete(force=True)")
